# Test execution when no papers in database


In [4]:
import json
from unittest.mock import MagicMock

from paper_scanner.core.database import PapersDatabase
from paper_scanner.steps.citations import CitationsStep

general_config = {}
tmp_path = "/tmp"

db = MagicMock(spec=PapersDatabase)
db.all.return_value = []
db.update = MagicMock()
db.save = MagicMock()

step = CitationsStep(general_config=general_config, db=db, cache_dir=tmp_path)


config = {
    "paper-types": ["journal_article"],
    "backward": {
        "source": ["crossref"],
        "continue_on_not_found": True
    }
}

results = step.execute(config, verbose=False, dry_run=True)

assert results["total_papers"] == 0
assert results["target_papers"] == 0
assert results["citations_fetched"] == 0

print(json.dumps(results, indent=2))

{
  "total_papers": 0,
  "target_papers": 0,
  "papers_with_citations": 0,
  "citations_fetched": 0,
  "citations_resolved": 0,
  "citations_created_new_paper": 0,
  "citations_unresolved": 0,
  "errors": [],
  "cache_hits": 0,
  "cache_misses": 0
}


In [5]:
from unittest.mock import MagicMock, patch

from paper_scanner.core.enum import DiscoveryMethod
from paper_scanner.core.models import Discovery, Paper, PaperType
from paper_scanner.steps.citations import CitationsStep

# Create mocks
general_config = {}
tmp_path = "/tmp"
mock_db = MagicMock()

# Create step instance
step = CitationsStep(general_config=general_config, db=mock_db, cache_dir=tmp_path)

# Create paper
paper1 = Paper(
    cite_key="test2020",
    title="Test Paper",
    doi="10.1234/test1",
    year=2020,
    paper_type=PaperType.JOURNAL_ARTICLE,
    discovery=Discovery(method=DiscoveryMethod.KEYWORD_SEARCH, iteration=0)
)
step.db.all.return_value = [paper1]

# Patch and run test
with patch("paper_scanner.steps.citations.Fetcher") as mock_fetcher_class:
    mock_fetcher = MagicMock()
    mock_fetcher.fetch_citations.return_value = ([], False)
    mock_fetcher_class.return_value = mock_fetcher

    config = {
        "paper-types": ["journal_article"],
        "backward": {
            "source": ["crossref"],
            "continue_on_not_found": True
        }
    }

    results = step.execute(config, verbose=False, dry_run=True)

    assert results["total_papers"] == 1
    assert results["target_papers"] == 1
    assert results["citations_fetched"] == 0

    print("✓ test_execute_with_papers_no_citations passed")
    print(json.dumps(results, indent=2))

✓ test_execute_with_papers_no_citations passed
{
  "total_papers": 1,
  "target_papers": 1,
  "papers_with_citations": 0,
  "citations_fetched": 0,
  "citations_resolved": 0,
  "citations_created_new_paper": 0,
  "citations_unresolved": 0,
  "errors": [],
  "cache_hits": 0,
  "cache_misses": 1
}


In [ ]:
from unittest.mock import MagicMock, patch

from paper_scanner.core.enum import DiscoveryMethod
from paper_scanner.core.models import Citation, Discovery, Paper, PaperType
from paper_scanner.steps.citations import CitationsStep

# Create mocks
general_config = {}
tmp_path = "/tmp"
mock_db = MagicMock()

# Create step instance
step = CitationsStep(general_config=general_config, db=mock_db, cache_dir=tmp_path)

# Setup mock database with paper
citing_paper = Paper(
    cite_key="citing2020",
    title="Citing Paper",
    doi="10.1234/citing",
    year=2020,
    paper_type=PaperType.JOURNAL_ARTICLE,
    discovery=Discovery(method=DiscoveryMethod.KEYWORD_SEARCH, iteration=0)
)
step.db.all.return_value = [citing_paper]

# Patch and run test
with patch("paper_scanner.steps.citations.Fetcher") as mock_fetcher_class:
    # Setup mock citations
    citation1 = Citation(
        doi="10.1234/cited1",
        title="Cited Paper 1",
        year=2019,
        extraction_method="crossref",
        confidence=0.95
    )
    citation2 = Citation(
        title="Cited Paper 2",
        year=2018,
        extraction_method="crossref",
        confidence=0.75
    )

    # Setup mock fetcher
    mock_fetcher = MagicMock()
    mock_fetcher.fetch_citations.return_value = ([citation1, citation2], False)
    mock_fetcher_class.return_value = mock_fetcher

    config = {
        "paper-types": ["journal_article"],
        "backward": {
            "source": ["crossref"],
            "continue_on_not_found": False
        }
    }

    results = step.execute(config, verbose=False, dry_run=True, debug=False)

    assert results["total_papers"] == 1
    assert results["target_papers"] == 1
    assert results["citations_fetched"] == 2
    assert results["papers_with_citations"] == 1

    print("✓ test_execute_with_citations passed")
    print(json.dumps(results, indent=2))